# Kvasir-SEG Morphology and Visual Audit

Computes morphology summaries and saves visual diagnostics.\n\nOutputs under `0_dataset_prep/out/visualizations` and `0_dataset_prep/out/metadata`.

In [1]:
import sys
from pathlib import Path

def _bootstrap_kvasir_seg_path() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / 'utils' / 'segmentation_common.py').exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p
        alt = p / 'Prototyping_reformat' / 'DatasetAnalysis' / 'Kvasir_SEG'
        if (alt / 'utils' / 'segmentation_common.py').exists():
            if str(alt) not in sys.path:
                sys.path.insert(0, str(alt))
            return alt
    raise RuntimeError('Could not locate Kvasir_SEG utils path from current working directory.')

BOOTSTRAP_ROOT = _bootstrap_kvasir_seg_path()
print('BOOTSTRAP_ROOT:', BOOTSTRAP_ROOT)

BOOTSTRAP_ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG


In [2]:

import json
from pathlib import Path
import numpy as np
import pandas as pd

from utils.segmentation_common import (
    find_kvasir_seg_root,
    load_metadata,
    save_histograms,
    save_overlay_grid,
)

ROOT = find_kvasir_seg_root()
META_CSV = ROOT / '0_dataset_prep' / 'out' / 'metadata' / 'metadata_enriched.csv'
VIZ_DIR = ROOT / '0_dataset_prep' / 'out' / 'visualizations'
META_DIR = ROOT / '0_dataset_prep' / 'out' / 'metadata'
VIZ_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('META_CSV:', META_CSV)


ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG
META_CSV: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/0_dataset_prep/out/metadata/metadata_enriched.csv


In [3]:

df = load_metadata(META_CSV)

summary = {
    'n_images': int(len(df)),
    'width_mean': float(df['width'].mean()),
    'height_mean': float(df['height'].mean()),
    'width_min': int(df['width'].min()),
    'width_max': int(df['width'].max()),
    'height_min': int(df['height'].min()),
    'height_max': int(df['height'].max()),
    'mask_area_ratio_mean': float(df['mask_area_ratio'].mean()),
    'mask_area_ratio_median': float(df['mask_area_ratio'].median()),
    'mask_area_ratio_p10': float(df['mask_area_ratio'].quantile(0.1)),
    'mask_area_ratio_p90': float(df['mask_area_ratio'].quantile(0.9)),
    'component_count_mean': float(df['component_count'].mean()),
    'component_count_median': float(df['component_count'].median()),
    'single_component_share': float((df['component_count'] <= 1).mean()),
    'multi_component_share': float((df['component_count'] > 1).mean()),
    'bbox_count_mean': float(df['bbox_count'].mean()),
    'bbox_count_max': int(df['bbox_count'].max()),
}


In [4]:

save_histograms(df, VIZ_DIR)
save_overlay_grid(df, VIZ_DIR / 'sample_overlay_grid.png', n=24, seed=42)

summary_path = META_DIR / 'morphology_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved histograms + overlay grid to:', VIZ_DIR)
print('Saved summary:', summary_path)
print(json.dumps(summary, indent=2))


Saved histograms + overlay grid to: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/0_dataset_prep/out/visualizations
Saved summary: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/0_dataset_prep/out/metadata/morphology_summary.json
{
  "n_images": 1000,
  "width_mean": 625.292,
  "height_mean": 545.228,
  "width_min": 332,
  "width_max": 1920,
  "height_min": 352,
  "height_max": 1072,
  "mask_area_ratio_mean": 0.15390989411452585,
  "mask_area_ratio_median": 0.1140072415670201,
  "mask_area_ratio_p10": 0.0341076209733541,
  "mask_area_ratio_p90": 0.3190950249035716,
  "component_count_mean": 1.207,
  "component_count_median": 1.0,
  "single_component_share": 0.819,
  "multi_component_share": 0.181,
  "bbox_count_mean": 1.071,
  "bbox_count_max": 10
}
